In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import math
import hist
import vector
import os
import subprocess
import gc
print("uproot version",uproot.__version__)
print("awkward version",ak.__version__)
print("numpy version",np.__version__)
print("matplotlib version",matplotlib.__version__)
print("hist version",hist.__version__)
print("vector version",vector.__version__)

uproot version 5.5.1
awkward version 2.7.2
numpy version 2.0.2
matplotlib version 3.9.0
hist version 2.8.0
vector version 1.5.2


In [2]:
vector.register_awkward()

In [3]:
file = uproot.open("/pbs/throng/training/nantes-m2-rps-exp/data/run291263.mc.root")  # runXXX.mc.root pour les donnees simulees

events = file["eventsTree"]
events.show()

name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
runNumber            | int32_t                  | AsDtype('>i4')
xVtx                 | double                   | AsDtype('>f8')
yVtx                 | double                   | AsDtype('>f8')
zVtx                 | double                   | AsDtype('>f8')
isCINT               | bool                     | AsDtype('bool')
isCMSL               | bool                     | AsDtype('bool')
isCMSH               | bool                     | AsDtype('bool')
isCMLL               | bool                     | AsDtype('bool')
isCMUL               | bool                     | AsDtype('bool')
nMuons               | int32_t                  | AsDtype('>i4')
Muon_E               | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
Muon_Px              | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
Muon_Py              

In [4]:
m = events.arrays(["nMuons","Muon_Px","Muon_Py","Muon_Pz","Muon_Charge","Muon_E","Muon_thetaAbs","Muon_matchedTrgThreshold"],entry_stop=10000)
print(m)

[{nMuons: 2, Muon_Px: [2.02, -0.71], Muon_Py: [...], Muon_Pz: [...], ...}, ...]


In [5]:
def getTracks(events):
    return ak.zip({"px":events["Muon_Px"],
                       "py":events["Muon_Py"],
                       "pz":events["Muon_Pz"],
                       "E":events["Muon_E"],
                       "charge":events["Muon_Charge"],
                       "thetaAbs":events["Muon_thetaAbs"],
                       "matched":events["Muon_matchedTrgThreshold"]},
                    with_name='Momentum4D')

In [6]:
def scan(dataDescription, 
              hMag:hist.Hist, hPhi:hist.Hist,
              eventSelector=lambda x:[True]*len(x),
              trackSelector=lambda x:[True]*len(x), 
              verbose:bool=False):
    """ Loop over data to fill the invariant mass histogram.
        
        :param: dataDescription: is anything uproot.iterate can take.
                typical something like run*.data.root:eventsTree in our case
        :param: eventSelector: returns an array of bool from an array of events
        :param: trackSelector: returns an array of bool from an array of tracks
    """
    
    for batch in uproot.iterate(dataDescription,
                                ["isCINT","isCMUL","isCMSL","Muon_Px","Muon_Py","Muon_Pz","Muon_E","Muon_Charge","Muon_thetaAbs","Muon_matchedTrgThreshold"],                                
                                 report=True):
        events=batch[0] # batch[1] is the report info
        if len(events) < 1000:
            print("something is wrong",batch[1]) # this is a protection for some corrupted input data files 
            break
            
        goodEvents = events[eventSelector(events)] 
        
        tracks = getTracks(events)
        goodTracks=tracks[trackSelector(tracks)]
    
        hMag.fill(ak.flatten(goodTracks.p))
        hPhi.fill(ak.flatten(goodTracks.phi))    

        if verbose:
            print(batch[1])
        gc.collect()

In [8]:
tracks = getTracks(m)
print(m)
print(events)
type(tracks[0])
print(len(tracks))
#print(tracks[0])
a= tracks[0][0] #une trace
b =tracks[0][1] #une trace

c = (a+b).M #un événement contenant 2 traces
print(c)

tracks[0].M #donne la masse invariante pour les différentes traces 
#for j in m[m.nMuons > 1]:
 #   for i in range(len(tracks)):
  #      print(tracks[i].M)


[{nMuons: 2, Muon_Px: [2.02, -0.71], Muon_Py: [...], Muon_Pz: [...], ...}, ...]
<TTree 'eventsTree' (25 branches) at 0x7fb42cfef850>
10000
3.6578522


<Array [0.106, 0.106] type='2 * float32'>

In [95]:
ak.where(m.nMuons ==2)

(<Array [0, 1, 3, 12, 13, ..., 9980, 9982, 9992, 9997, 9998] type='2910 * int64'>,)

In [51]:
m[0].to_list()

{'nMuons': 2,
 'Muon': [{'Px': 2.0223944187164307,
   'Py': -0.9315006732940674,
   'Pz': -30.101959228515625,
   'Charge': 1,
   'E': 30.18438148498535},
  {'Px': -0.7104543447494507,
   'Py': -3.694413423538208,
   'Pz': -66.21504211425781,
   'Charge': -1,
   'E': 66.32191467285156}]}

In [ ]:
energie_muons = np.array([],dtype=float)
nofTracks=0 # il est toujours utile de compter ...
nofEvents=0 # 
for event in m[m.nMuons>1]:
    nofEvents+=1
    tracks = event["Muon"].to_list()
    for t in tracks:
        nofTracks+=1
        energie_muons = np.append(energie_muons,t["Px"],t["Py"],t["Pz"]))

        
        

In [54]:
for event in m[m.nMuons>1]:
    tracks = event["Muon"].to_list()
print(tracks)


[{'Px': -0.3215143084526062, 'Py': -0.9697946310043335, 'Pz': -9.542024612426758, 'Charge': -1, 'E': 9.597148895263672}, {'Px': 3.1425044536590576, 'Py': 0.2220907360315323, 'Pz': -22.86820411682129, 'Charge': 1, 'E': 23.084421157836914}]


In [ ]:
import awkward as ak
import vector

# Filtrer les événements avec nMuons == 2 et isCMUL == True
#attention lors d'une coupure à regarder le nb d'event avant et après pour savoir si on supprime pas trop d'event
filtered_events = m[(m.nMuons > 1) & (m.isCMUL)&(abs(m.zVtx)<10)&(m.isCMSH)] #ici on fait les coupures sur les événements.
# Initialiser les vecteurs à partir des données des muons
muon_vectors = ak.zip({
    "px": filtered_events.Muon.Px,
    "py": filtered_events.Muon.Py,
    "pz": filtered_events.Muon.Pz,
    "E": filtered_events.Muon.E
}, with_name="Momentum4D")

# Calculer les masses invariantes pour chaque paire de muons dans chaque événement
invariant_masses = ak.combinations(muon_vectors, 2, fields=["muon1", "muon2"])
masses = (invariant_masses.muon1 + invariant_masses.muon2).mass

# Créer un histogramme des valeurs de la masse
flattened_masses = ak.flatten(masses)  # Aplatir les données pour l'histogramme
filtered_masses = flattened_masses[(flattened_masses >= 0) & (flattened_masses <= 5)]  # Filtrer les masses entre 0 et 10
plt.hist(filtered_masses, bins=500, color='blue', alpha=0.7)
plt.xlabel('Masse invariante (GeV/c^2)')
plt.ylabel("Nombre d'événements")
plt.title('Distribution des masses invariantes des paires de muons (0-5 GeV/c^2)')
plt.grid(True)
plt.show()